# Задача № 3. Военный или гражданский самолёт — Google Colab

**Бесплатная редакция: без HF-токена, платных API и установки программ на компьютер.**
Две открытые мультимодальные LLM скачиваются с Hugging Face и выполняются на GPU среды Colab:
**SmolVLM-500M-Instruct** и **Qwen2.5-VL-3B-Instruct**.

**Статус:** данные и код подготовлены; реальный прогон этих моделей ещё не выполнен.
Таблицы качества и вывод формируются только по 60 реально полученным ответам.

> Способ запуска отличается от исходного упоминания «провайдеров LLM»: используется Hugging Face Hub
> как источник моделей и Colab как вычислительная среда, а не HF Inference Providers.
> Это выбранная пользователем бесплатная редакция; при строгом требовании провайдеров её нужно согласовать с преподавателем.

## Как пользоваться
1. Сохраните копию документа на Google Диске.
2. **Среда выполнения → Сменить среду выполнения → T4 GPU** (если бесплатный GPU доступен).
3. **Среда выполнения → Выполнить всё**. Не нужно вводить токен, загружать ZIP проекта или править код.
4. Дождитесь метрик и скачайте ZIP результатов в последней ячейке. Сохраните документ с выводами.

Библиотеки и модели устанавливаются/скачиваются только во временную среду Google, не на ваш компьютер.
Первый запуск скачивает несколько гигабайт весов. Модели загружаются последовательно.
Бесплатный GPU не гарантирован, квота Colab ограничена: если его не выдали, попробуйте позже.
CPU тоже поддерживается, но сильно медленнее; он включается отдельной настройкой `ALLOW_CPU=True`.

Расширение `.ipynb` — формат этого документа Colab. Устанавливать Jupyter или Anaconda не требуется.

In [ ]:
# 1. Загрузить проект из GitHub автоматически. Токен GitHub не нужен: репозиторий открыт.
import io, json, os, sys, urllib.request, zipfile
from pathlib import Path

os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
ROOT = Path('/content/aircraft-hf-comparison')
project_ready = ((ROOT / 'src/experiment.py').exists() and (ROOT / 'config.json').exists()
                 and json.loads((ROOT / 'config.json').read_text()).get('execution') == 'colab_local')
if not project_ready:
    url = 'https://codeload.github.com/Youdzter-Utka/ISIT-work-3/zip/refs/heads/main'
    request = urllib.request.Request(url, headers={'User-Agent': 'AircraftColabCourse/2.0'})
    with urllib.request.urlopen(request, timeout=120) as response:
        archive_bytes = response.read()
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        for member in archive.infolist():
            parts = Path(member.filename).parts
            if len(parts) < 2 or member.is_dir():
                continue
            relative = Path(*parts[1:])
            target = (ROOT / relative).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise ValueError('Недопустимый путь в архиве')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
    print('Проект и 30 фотографий загружены.')
else:
    print('Проект уже загружен; сохранённые ответы остаются на месте.')
os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
# 2. Проверить вычислительную среду. Бесплатного GPU T4 достаточно для последовательного запуска.
import torch
ALLOW_CPU = False  # Обычно оставьте False. True — только для медленного CPU-запуска.
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError('GPU не выделен. Выберите T4 GPU в настройках среды Colab. Если квота недоступна, попробуйте позже.')
print('Устройство:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (медленно)')

In [ ]:
# 3. Установить библиотеки в Colab. PyTorch и torchvision сохраняются из готовой среды Google.
%pip -q install -r /content/aircraft-hf-comparison/requirements.txt

## 1. Постановка задачи и эталонные ответы
Нужно по фото определить, военный самолёт или гражданский, и сравнить две модели на одних и тех же данных.
Обучения и дообучения нет: zero-shot, то есть модель получает фотографию и инструкцию без примеров ответов.

Эталон — заранее заданное **основное назначение изображённой модификации**, а не владелец борта:
- `military`: истребители, штурмовики, бомбардировщики, военные учебные, транспортные, заправщики и патрульные;
- `civilian`: пассажирские, гражданские грузовые, частные, спортивные, сельскохозяйственные и пожарные.

30 различных типов: 15 каждого класса; 20 заранее отмечены как сложные, ещё 10 — контрольные.
Военные заправщики похожи на авиалайнеры, гражданский L-100 происходит от военного Hercules,
спортивный самолёт может напоминать военный учебный. Названия и правильные ответы моделям не передаются.
На фото 09 и 10 классифицируется самый крупный самолёт, остальные борта — контекст дозаправки.
Фото 21 изображает гражданскую конструкцию L-100 у военного оператора; по нашему правилу это civilian.
Метки и сложность зафиксированы до получения предсказаний.

In [ ]:
# 4. Проверить 30 фотографий, эталон и контрольные суммы.
import json
import pandas as pd
from IPython.display import display, HTML, Image, Markdown
import importlib, experiment
importlib.reload(experiment)
from experiment import read_manifest, load_config, run, report, PROMPT, read_log

rows = read_manifest()
manifest = pd.DataFrame(rows)
display(manifest[['id', 'page_title', 'label', 'hard', 'reason']])
display(pd.crosstab(manifest.label, manifest.hard))
print('Проверено:', len(rows), 'фото, по 15 в каждом классе; 20 сложных.')

### Галерея и источники
Все JPEG и атрибуция включены в репозиторий. Источники — Wikipedia и Wikimedia Commons.
Фотографии были визуально проверены вместе с описаниями до обращения к моделям.
Фотографии уменьшены и перекодированы в JPEG; авторы, лицензии, ссылки и SHA-256 — в `data/manifest.csv`.
В галерее можно отдельно проверить назначение типа и страницу исходного фото.

In [ ]:
# 5. Показать всю выборку. Названия здесь видит читатель отчёта, но не модель.
from html import escape
for row in rows:
    display(HTML(f"<h4>{escape(row['id'])}: {escape(row['page_title'])} — {escape(row['label'])}</h4>"
        f"<p>{escape(row['reason'])}</p><a href='{escape(row['article_url'])}'>Назначение типа</a> · "
        f"<a href='{escape(row['file_url'])}'>Источник и лицензия</a><p>{escape(row['license'])}</p>"))
    display(Image(filename=row['image_path'], width=480))

01: General Dynamics F-16 Fighting Falcon — military Типичный истребитель Назначение · Источник и лицензия Public domain

02: Sukhoi Su-27 — military Типичный истребитель Назначение · Источник и лицензия GFDL 1.2

03: Lockheed Martin F-35 Lightning II — military Типичный истребитель Назначение · Источник и лицензия Public domain

04: Northrop B-2 Spirit — military Бомбардировщик необычной формы Назначение · Источник и лицензия OGL v1.0

05: Fairchild Republic A-10 Thunderbolt II — military Штурмовик Назначение · Источник и лицензия Public domain

...

26: Bede BD-5 — civilian Спортивный, необычный силуэт Назначение · Источник и лицензия CC0

27: Pilatus PC-12 — civilian Гражданский тип со специальными операторами Назначение · Источник и лицензия GFDL 1.2

28: Piaggio P.180 Avanti — civilian Необычная компоновка Назначение · Источник и лицензия CC BY-SA 3.0

29: Airbus Beluga — civilian Специальный грузовой Назначение · Источник и лицензия CC BY-SA 4.0

30: Boeing Dreamlifter — civilian Крупногабаритный грузовой Назначение · Источник и лицензия CC BY-SA 2.0

## 2. Модели и честное сравнение
| Модель | Зачем выбрана |
|---|---|
| SmolVLM-500M-Instruct | Компактная модель с небольшими требованиями к памяти |
| Qwen2.5-VL-3B-Instruct | Более крупная модель другого семейства; проверяем, даст ли это выигрыш |

Версии моделей закреплены SHA коммитов в `config.json`. Оба репозитория открыты, скачивание выполняется
с `token=False`. Платных запросов к HF Inference Providers нет: ошибка баланса 402 этого API больше не участвует в запуске.

Обе модели получают одинаковые исходные JPEG, ограниченные по длинной стороне 768 px, и одну инструкцию.
Дальнейшие преобразования штатных процессоров моделей различаются и зафиксированы в коде:
у SmolVLM longest_edge=1024; у Qwen min_pixels=256×28² и max_pixels=512×28².
Это сравнение готовых модельных систем, а не одинаковых архитектур с одним процессором.

Генерация без случайного выбора (`do_sample=False`, максимум 16 новых токенов), без обучения и подбора промпта
по тестовым результатам. Просим только одно слово, чтобы формат JSON не усложнял задачу небольшой модели.
При разборе допускаются регистр, внешние кавычки и завершающая точка. Другие ответы — `invalid`.
Случайность ограничена seed=42, но побитовая воспроизводимость на разных GPU не гарантируется.

In [ ]:
# 6. Посмотреть параметры и одинаковый для обеих моделей промпт.
display(load_config())
print(PROMPT)

## 3. Необязательное восстановление результатов
Обычно эту ячейку менять не нужно. Если предыдущая сессия Colab завершилась,
а вы успели скачать ZIP результатов, установите `RESTORE_RESULTS=True` и загрузите этот ZIP.
Версии библиотек, GPU, модели, фото и промпт должны совпасть; смешивание разных экспериментов запрещено.

In [ ]:
# 7. Для первого запуска оставьте False.
RESTORE_RESULTS = False
if RESTORE_RESULTS:
    from google.colab import files
    if (ROOT / 'results/predictions.jsonl').exists():
        raise RuntimeError('Ответы уже есть в этой среде; восстановление поверх них запрещено.')
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Выберите один ZIP результатов.')
    content = next(iter(uploaded.values()))
    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        for member in archive.infolist():
            relative = Path(member.filename)
            if member.is_dir():
                continue
            if not relative.parts or relative.parts[0] != 'results':
                continue
            target = (ROOT / relative).resolve()
            if not target.is_relative_to((ROOT / 'results').resolve()):
                raise ValueError('Недопустимый путь в архиве')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(member))
    print('Результаты восстановлены. Следующая ячейка проверит совместимость и продолжит.')

## 4. Выполнение: 30 фото × 2 модели
Первый запуск загружает веса из интернета. Сначала обрабатываются 30 фото SmolVLM,
после освобождения памяти — 30 фото Qwen. Порядок выбран ради экономии памяти, поэтому
сравнение времени может зависеть от нагрузки Colab и прогрева устройства.

Каждый ответ сразу добавляется в `results/predictions.jsonl`. Повторный запуск в той же среде
пропускает готовые ответы. Ошибки загрузки/памяти не превращаются в вымышленные предсказания:
выполнение останавливается, предыдущие строки остаются. Для получения всех результатов продолжите запуск.
Если среда отключится полностью, её временные файлы могут исчезнуть — скачивайте архив результатов.

In [ ]:
# 8. Реальный запуск. HF_TOKEN и секреты не нужны.
# По умолчанию выполняются все 60 предсказаний; для пробного запуска можно передать limit=2.
run_status = run(allow_cpu=ALLOW_CPU)

## 5. Метрики и матрицы ошибок
Основная метрика — **macro-F1**: среднее F1 двух классов. Дополнительно считаются:
- accuracy — доля правильных ответов;
- macro-precision и macro-recall, а также precision/recall/F1 для каждого класса;
- accuracy на 20 сложных случаях;
- coverage — доля ответов в допустимом формате;
- средняя и медианная задержка обработки изображения.

`invalid` считается ошибкой и false negative истинного класса; он не удаляется из знаменателя.
Матрица ошибок содержит отдельную колонку invalid. Задержка включает обработку изображения и генерацию,
но не загрузку модели и весов из интернета; GPU синхронизируется перед измерением и после генерации.
Первый пример может включать эффект прогрева. Время загрузки модели сохранено отдельно.

Для парного сравнения правильности используется точный тест Мак-Немара. На 30 фото одна ошибка
меняет accuracy на 3,33 процентного пункта; малую разницу нельзя автоматически считать устойчивым превосходством.

In [ ]:
# 9. Сравнение строится только по полным и проверенным данным.
records = read_log(ROOT / 'results/predictions.jsonl')
results_ready = len(records) == 60
if results_ready:
    metrics, errors, summary = report()
    display(metrics.round(4))
    display(pd.read_csv('results/per_class_metrics.csv').round(4))
    display(Image(filename='results/confusion_matrices.png'))
else:
    print(f'Получено {len(records)}/60 ответов. Выполните ячейку запуска ещё раз.')

## 6. Разбор ошибок
Для каждой ошибки приведены самолёт, эталон, ответ модели, исходный текст и причина сложности.
Наиболее интересны военные транспортные/учебные и гражданские самолёты, похожие на военные.
Ниже выводятся все ошибки и первые три фотографии с ошибками. После выполнения прокомментируйте
2–3 конкретных случая: какой визуальный признак мог ввести модель в заблуждение.

In [ ]:
# 10. Реальные ошибки; этот блок ничего не придумывает при отсутствии предсказаний.
if results_ready:
    if errors.empty:
        print('На этих 30 фото обе модели ответили правильно.')
    else:
        display(errors[['id', 'page_title', 'model', 'truth', 'prediction', 'raw', 'reason']])
        for _, error in errors.drop_duplicates('id').head(3).iterrows():
            display(Markdown(f"**{error['id']}: {error['page_title']}** — эталон {error['truth']}. {error['reason']}"))
            display(Image(filename=error['image_path'], width=480))

## 7. Вывод по выполненному эксперименту
Вывод ниже формируется из реальных метрик. До полного запуска победитель не объявляется.

In [ ]:
# 11. Автоматический численный вывод.
if results_ready:
    best, other = metrics.iloc[0], metrics.iloc[1]
    if abs(best.f1_macro - other.f1_macro) < 1e-12:
        conclusion = 'Модели получили одинаковый macro-F1.'
    else:
        conclusion = (f"На выбранных 30 фото выше macro-F1 у **{best['model']}**: "
                      f"{best.f1_macro:.3f} против {other.f1_macro:.3f}.")
    display(Markdown(conclusion + f" Точный тест Мак-Немара: p={summary['mcnemar_exact_p']:.4f}. "
        'Это результат небольшой целевой выборки; обобщать его на все фото самолётов нельзя.'))
else:
    print('Численный вывод появится после завершения всех 60 предсказаний.')

## 8. Сохранение для сдачи и продолжения
Последняя ячейка скачает ZIP с текущими результатами, конфигурацией и инструкцией.
Если получено меньше 60 ответов, архив всё равно полезен для продолжения в совместимой среде.
Внутри ZIP находятся ответы и результаты; веса моделей в архив не входят.
Сохраните также **сам выполненный документ Colab** через «Файл → Скачать → Скачать файл IPYNB»
или сохраните его на Google Диске. Ссылка на публичный отчёт должна давать преподавателю доступ на чтение.

In [ ]:
# 12. Скачать ответы, таблицы и график. В Colab откроется обычное скачивание файла.
from google.colab import files
archive_path = Path('/content/aircraft-results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((ROOT / 'results').rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(ROOT).as_posix())
    archive.write(ROOT / 'config.json', 'config.json')
    archive.write(ROOT / 'README.md', 'README.md')
files.download(str(archive_path))

## Ограничения и воспроизводимость
- Выборка намеренно сложная и сбалансированная, а не случайная выборка реальных авиационных фотографий.
- Открытые фото могли попасть в обучающие данные моделей; отсутствие утечки не гарантируется.
- Назначение модификации иногда неоднозначно по одному кадру. Эталон задан по правилу выше.
- У моделей разные размеры, обучение, лицензии и процессоры; сравнение не изолирует влияние одного фактора.
- В журнале сохраняются SHA моделей и JPEG, промпт, версии библиотек, тип GPU, dtype, параметры и UTC-время.
- Все 60 ответов должны принадлежать одной подписи эксперимента. Подмена меток и смешивание прогонов проверяются.
- Бесплатные ресурсы Colab ограничены и не гарантированы. Скачивание весов также зависит от сети и доступности HF.

## Источники
1. [SmolVLM-500M-Instruct](https://huggingface.co/HuggingFaceTB/SmolVLM-500M-Instruct)
2. [Qwen2.5-VL-3B-Instruct](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)
3. [SmolVLM в Transformers 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/model_doc/smolvlm)
4. [Qwen2.5-VL в Transformers 4.57.1](https://huggingface.co/docs/transformers/v4.57.1/model_doc/qwen2_5_vl)
5. [Colab: бесплатный доступ и ограничения](https://research.google.com/colaboratory/faq.html)
6. Индивидуальные авторы, лицензии и ссылки фотографий — `data/manifest.csv`.

Проверка открытого доступа к моделям без авторизации: 24.10.2026.

## Приложение. Полный код эксперимента

Этот код запускается из `src/experiment.py`; он приведён здесь для чтения и проверки.

```python
"""Offline inference on downloaded HF models; intended for Google Colab."""
from __future__ import annotations

import gc
import hashlib
import json
import platform
import re
import time
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
LABELS = ["military", "civilian"]
PROMPT = (
    "Classify the largest, primary aircraft in this photo. "
    "Use the main intended role of its aircraft variant, not only its operator. "
    "Military includes fighters, military trainers, transports, tankers and patrol aircraft. "
    "Civilian includes passenger, civilian cargo, private, sport, agricultural and firefighting aircraft. "
    "Reply with exactly one word: military or civilian."
)


def read_manifest():
    """Validate the frozen 30-image dataset before inference or evaluation."""
    import csv

    with (ROOT / "data/manifest.csv").open(encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != 30 or len({row["id"] for row in rows}) != 30:
        raise ValueError("The dataset must contain 30 unique IDs.")
    if any(sum(row["label"] == label for row in rows) != 15 for label in LABELS):
        raise ValueError("Expected 15 military and 15 civilian images.")
    hashes = []
    for row in rows:
        if row["reviewed"] != "1" or row["hard"] not in ("0", "1"):
            raise ValueError("Unreviewed or invalid annotation: " + row["id"])
        path = (ROOT / row["image_path"]).resolve()
        if not path.is_relative_to((ROOT / "data/images").resolve()):
            raise ValueError("Image path outside the dataset.")
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != row["sha256"]:
            raise ValueError("Image changed: " + row["id"])
        hashes.append(digest)
    if len(set(hashes)) != 30:
        raise ValueError("Duplicate photos found.")
    return rows


def load_config():
    config = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
    models = config["models"]
    if len(models) != 2 or len({model["id"] for model in models}) != 2:
        raise ValueError("Exactly two different models are required.")
    for model in models:
        if not re.fullmatch(r"[0-9a-f]{40}", model["revision"]):
            raise ValueError("Pin each model to a full Hub commit SHA.")
    if config["max_new_tokens"] < 1 or config["max_image_edge"] < 1:
        raise ValueError("Invalid generation settings.")
    return config


def experiment_id(config, rows, runtime):
    """Include model revisions, image hashes, prompt and inference environment."""
    payload = {
        "config": config,
        "prompt": PROMPT,
        "data": [
            {key: row[key] for key in ("id", "sha256", "label", "hard")}
            for row in rows
        ],
        "runtime": runtime,
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def parse_label(raw):
    """Accept a single class name with optional final period or outer quotes."""
    label = raw.strip().lower().strip('"\'').removesuffix(".").strip()
    return label if label in LABELS else "invalid"


def read_log(path):
    if not path.exists():
        return []
    records = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        try:
            record = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f"Corrupt log line {line_number}; restore a backup.") from exc
        records.append(record)
    keys = [(record["model"], record["id"]) for record in records]
    if len(keys) != len(set(keys)):
        raise ValueError("Duplicate predictions in log.")
    return records


def runtime_info(torch):
    packages = {}
    for package in ("torch", "transformers", "accelerate", "Pillow", "huggingface_hub"):
        try:
            packages[package] = version(package)
        except PackageNotFoundError:
            packages[package] = "unavailable"
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return {
        "python": platform.python_version(),
        "packages": packages,
        "device": device,
        "device_name": torch.cuda.get_device_name(0) if device == "cuda" else "CPU",
        "dtype": "float16" if device == "cuda" else "float32",
        "attention": "sdpa",
    }


def _load_model(spec, device, dtype):
    from transformers import AutoModelForImageTextToText, AutoProcessor

    # token=False prevents use of any cached HF credentials. Both repos are public.
    kwargs = {"revision": spec["revision"], "token": False, "trust_remote_code": False}
    processor_kwargs = {}
    if spec["family"] == "qwen2_5_vl":
        processor_kwargs = {"min_pixels": 256 * 28 * 28, "max_pixels": 512 * 28 * 28}
    processor = AutoProcessor.from_pretrained(spec["id"], **kwargs, **processor_kwargs)
    if spec["family"] == "smolvlm":
        # Documented SmolVLM image processor option limits image splitting.
        processor.image_processor.size = {"longest_edge": 1024}
    model = AutoModelForImageTextToText.from_pretrained(
        spec["id"], **kwargs, torch_dtype=dtype,
        attn_implementation="sdpa", low_cpu_mem_usage=True,
    ).to(device).eval()
    return processor, model


def _synchronize(torch, device):
    if device == "cuda":
        torch.cuda.synchronize()


def _predict(processor, model, row, config, torch, device, dtype):
    from PIL import Image

    with Image.open(ROOT / row["image_path"]) as source:
        image = source.convert("RGB")
    image.thumbnail((config["max_image_edge"], config["max_image_edge"]))
    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": PROMPT},
    ]}]
    # Only photo pixels and PROMPT enter the model. No filename, URL or truth label.
    text = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    _synchronize(torch, device)
    start = time.perf_counter()
    inputs = processor(text=[text], images=[image], return_tensors="pt", padding=True)
    inputs = inputs.to(device)
    # Cast only floating tensors; token IDs and grid dimensions must remain integers.
    for key, value in inputs.items():
        if torch.is_floating_point(value):
            inputs[key] = value.to(dtype=dtype)
    with torch.inference_mode():
        output = model.generate(
            **inputs, do_sample=False, max_new_tokens=config["max_new_tokens"],
            use_cache=True,
        )
    _synchronize(torch, device)
    latency = time.perf_counter() - start
    # Decode ONLY new tokens; otherwise the prompt itself would contaminate parsing.
    new_tokens = output[:, inputs["input_ids"].shape[1]:]
    raw = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    generated_tokens = int(new_tokens.shape[1])
    return raw, latency, generated_tokens


def run(limit=None, allow_cpu=False):
    """Load one model at a time; save each answer and resume without repeating it."""
    import torch

    rows = read_manifest()
    config = load_config()
    if limit is not None and (not isinstance(limit, int) or limit < 1):
        raise ValueError("limit must be a positive integer.")
    runtime = runtime_info(torch)
    device = runtime["device"]
    if device == "cpu" and not allow_cpu:
        raise RuntimeError(
            "GPU не выделен. В Colab: Среда выполнения → Сменить среду выполнения → T4 GPU. "
            "Если бесплатная квота недоступна, попробуйте позже. "
            "Для осознанного медленного запуска на CPU: run(allow_cpu=True)."
        )
    dtype = torch.float16 if device == "cuda" else torch.float32
    torch.manual_seed(config["seed"])
    if device == "cuda":
        torch.cuda.manual_seed_all(config["seed"])
    torch.backends.cudnn.benchmark = False
    signature = experiment_id(config, rows, runtime)
    results = ROOT / "results"
    results.mkdir(exist_ok=True)
    log_path = results / "predictions.jsonl"
    records = read_log(log_path)
    if any(record["experiment"] != signature for record in records):
        raise ValueError("Settings, data or runtime changed. Archive previous results first.")
    done = {(record["model"], record["id"]) for record in records}
    expected_keys = {(spec["id"], row["id"]) for spec in config["models"] for row in rows}
    if not done <= expected_keys:
        raise ValueError("Unexpected model or image in saved predictions.")
    session = {"experiment": signature, "config": config, "prompt": PROMPT, "runtime": runtime}
    session_path = results / "experiment.json"
    session_path.write_text(json.dumps(session, ensure_ascii=False, indent=2), encoding="utf-8")
    (results / "status.json").write_text(json.dumps({
        "status": "complete" if done == expected_keys else "running",
        "predictions_obtained": len(done), "planned_predictions": 60,
    }, indent=2), encoding="utf-8")
    count = 0
    for spec in config["models"]:
        pending = [row for row in rows if (spec["id"], row["id"]) not in done]
        if not pending:
            continue
        if limit is not None and count >= limit:
            break
        print("Загрузка модели:", spec["id"], flush=True)
        processor = model = None
        try:
            load_start = time.perf_counter()
            processor, model = _load_model(spec, device, dtype)
            load_seconds = time.perf_counter() - load_start
            for row in pending:
                if limit is not None and count >= limit:
                    break
                raw, seconds, token_count = _predict(
                    processor, model, row, config, torch, device, dtype,
                )
                prediction = parse_label(raw)
                record = {
                    "experiment": signature, "id": row["id"], "model": spec["id"],
                    "revision": spec["revision"], "truth": row["label"],
                    "hard": int(row["hard"]), "sha256": row["sha256"],
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                    "raw": raw, "prediction": prediction,
                    "status": "invalid" if prediction == "invalid" else "ok",
                    "latency_s": seconds, "generated_tokens": token_count,
                    "model_load_s": load_seconds, "device": device,
                }
                # Each line is flushed on close; only a complete response is recorded.
                with log_path.open("a", encoding="utf-8") as handle:
                    handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                done.add((spec["id"], row["id"]))
                count += 1
                print(f"{len(done):02}/60 · {row['id']} · {spec['id']} → {prediction}", flush=True)
        finally:
            del processor, model
            gc.collect()
            if device == "cuda":
                torch.cuda.empty_cache()
    complete = done == expected_keys
    status = {"status": "complete" if complete else "incomplete",
              "predictions_obtained": len(done), "planned_predictions": 60}
    (results / "status.json").write_text(json.dumps(status, indent=2), encoding="utf-8")
    print("Готово: 60/60." if complete else f"Сохранено {len(done)}/60. Повторите run() для продолжения.")
    return status


def report():
    """Evaluate only a complete, consistent set of 60 real saved predictions."""
    import matplotlib.pyplot as plt
    import pandas as pd
    from math import comb
    from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

    rows = read_manifest()
    config = load_config()
    results = ROOT / "results"
    records = read_log(results / "predictions.jsonl")
    expected = {(spec["id"], row["id"]) for spec in config["models"] for row in rows}
    if {(record["model"], record["id"]) for record in records} != expected:
        raise ValueError(f"Для сравнения нужны все 60 ответов; сейчас {len(records)}.")
    session = json.loads((results / "experiment.json").read_text(encoding="utf-8"))
    signature = experiment_id(config, rows, session["runtime"])
    if session["experiment"] != signature or any(r["experiment"] != signature for r in records):
        raise ValueError("Predictions belong to a different experiment.")
    annotation = {row["id"]: row for row in rows}
    revisions = {spec["id"]: spec["revision"] for spec in config["models"]}
    for record in records:
        row = annotation[record["id"]]
        if (record["truth"] != row["label"] or record["sha256"] != row["sha256"]
                or record["hard"] != int(row["hard"])
                or record["revision"] != revisions[record["model"]]
                or record["prediction"] != parse_label(record["raw"])):
            raise ValueError("Saved response does not match frozen data or parsing rules.")
    df = pd.DataFrame(records)
    metrics, per_class = [], []
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, spec in zip(axes, config["models"]):
        model_id = spec["id"]
        group = df[df.model == model_id]
        precision, recall, f1, support = precision_recall_fscore_support(
            group.truth, group.prediction, labels=LABELS, zero_division=0,
        )
        metrics.append({
            "model": model_id, "n": len(group),
            "accuracy": accuracy_score(group.truth, group.prediction),
            "precision_macro": float(precision.mean()), "recall_macro": float(recall.mean()),
            "f1_macro": float(f1.mean()), "coverage": float(group.prediction.isin(LABELS).mean()),
            "hard_accuracy": float((group[group.hard == 1].truth == group[group.hard == 1].prediction).mean()),
            "mean_latency_s": group.latency_s.mean(), "median_latency_s": group.latency_s.median(),
        })
        for i, label in enumerate(LABELS):
            per_class.append({"model": model_id, "class": label, "precision": precision[i],
                              "recall": recall[i], "f1": f1[i], "support": int(support[i])})
        cm = confusion_matrix(group.truth, group.prediction, labels=LABELS + ["invalid"])[:2, :]
        ax.imshow(cm, cmap="Blues")
        ax.set_xticks(range(3), LABELS + ["invalid"])
        ax.set_yticks(range(2), LABELS)
        ax.set_xlabel("Prediction")
        ax.set_ylabel("Truth")
        ax.set_title(model_id.split("/")[-1])
        for y in range(2):
            for x in range(3):
                ax.text(x, y, str(cm[y, x]), ha="center", va="center")
    fig.tight_layout()
    fig.savefig(results / "confusion_matrices.png", dpi=160)
    plt.close(fig)
    table = pd.DataFrame(metrics).sort_values("f1_macro", ascending=False)
    table.to_csv(results / "metrics.csv", index=False)
    pd.DataFrame(per_class).to_csv(results / "per_class_metrics.csv", index=False)
    details = df.merge(pd.DataFrame(rows)[["id", "page_title", "reason", "file_url", "image_path"]], on="id")
    details.to_csv(results / "all_predictions.csv", index=False)
    errors = details[details.truth != details.prediction]
    errors.to_csv(results / "errors.csv", index=False)
    a, b = [spec["id"] for spec in config["models"]]
    pivot = df.pivot(index="id", columns="model", values="prediction")
    truth = df[df.model == a].set_index("id").truth
    correct_a, correct_b = pivot[a].eq(truth), pivot[b].eq(truth)
    n10 = int((correct_a & ~correct_b).sum())
    n01 = int((~correct_a & correct_b).sum())
    n = n10 + n01
    pvalue = min(1.0, 2 * sum(comb(n, k) for k in range(min(n10, n01) + 1)) / 2**n) if n else 1.0
    summary = {"status": "complete", "n_images": 30, "n_predictions": 60,
               "model_a": a, "model_b": b, "a_only_correct": n10,
               "b_only_correct": n01, "mcnemar_exact_p": pvalue}
    (results / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(table.to_string(index=False))
    return table, errors, summary

```